In [1]:
!pip install -q -U transformers peft datasets tqdm bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
# Setup device and imports
import json
import random
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from datasets import Dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using active device: {DEVICE}")

Using active device: cuda


In [3]:
# Serialization & Validation Helpers
def serialize_grid(grid):
    if not grid or not grid[0]:
        return ""
    return "\n".join(" ".join(str(cell) for cell in row) for row in grid)

def deserialize_grid(text_str):
    grid = []
    for line in text_str.strip().split("\n"):
        row = [int(x) for x in line.strip().split() if x.isdigit()]
        if row:
            grid.append(row)
    return grid if grid else [[0]]

def is_valid_grid(grid):
    try:
        import numpy as np
        arr = np.array(grid)
        if arr.ndim != 2 or arr.size == 0 or any(not 0 < s <= 30 for s in arr.shape):
            return False
        if arr.min() < 0 or arr.max() > 9:
            return False
        return True
    except Exception:
        return False

In [4]:
# Data Augmentation
import numpy as np

def rotate_grid(grid, k):
    return np.rot90(np.array(grid), k).tolist()

def flip_grid(grid, mode='horizontal'):
    arr = np.array(grid)
    if mode == 'horizontal':
        return np.fliplr(arr).tolist()
    return np.flipud(arr).tolist()

def permute_colors(grid, color_map):
    return [[color_map.get(cell, cell) for cell in row] for row in grid]

def generate_augmentations(train_pairs, num_augmentations=30):
    augmented = [(p['input'], p['output']) for p in train_pairs]
    while len(augmented) < num_augmentations:
        pair = random.choice(train_pairs)
        inp, out = pair['input'], pair['output']
        op = random.choice(['rot', 'flip', 'color'])
        if op == 'rot':
            k = random.choice([1, 2, 3])
            aug_in, aug_out = rotate_grid(inp, k), rotate_grid(out, k)
        elif op == 'flip':
            mode = random.choice(['horizontal', 'vertical'])
            aug_in, aug_out = flip_grid(inp, mode), flip_grid(out, mode)
        else:
            colors = list(range(10))
            shuffled = colors.copy()
            random.shuffle(shuffled)
            cmap = dict(zip(colors, shuffled))
            aug_in, aug_out = permute_colors(inp, cmap), permute_colors(out, cmap)
        augmented.append((aug_in, aug_out))
    return augmented[:num_augmentations]

In [ ]:
# Load Model with QLoRA for Kaggle GPU
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

def load_qlora_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4"
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto"
    )
    
    model = prepare_model_for_kbit_training(model)
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, peft_config)
    return model, tokenizer

model, tokenizer = load_qlora_model()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Run TTT on a Sample Task & Save Weights 
import os

# Load sample dataset task
arc_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json"
with open(arc_path, 'r') as f:
    tasks = json.load(f)

sample_id = list(tasks.keys())[0]
task = tasks[sample_id]

# Generate Training Dataset via Augmentations
augmented = generate_augmentations(task["train"], num_augmentations=30)
prompts = [f"### Input Grid:\n{serialize_grid(inp)}\n\n### Output Grid:\n{serialize_grid(out)}" for inp, out in augmented]
dataset = Dataset.from_dict({"text": prompts})

def tokenize_fn(examples):
    tokens = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_fn, batched=True)

# Fine-tune with Trainer
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=TrainingArguments(
        output_dir="./saved_arc_adapter",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=1,
        fp16=True,
        logging_steps=5,
        save_strategy="epoch",
        report_to="none"
    )
)

print("Starting Test-Time Training...")
model.train()
trainer.train()

# Save the trained QLoRA adapter weights cleanly
output_dir = "./final_arc_adapter"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Model adapter successfully saved to {output_dir}")

In [ ]:
# Create a zip archive 
import shutil
shutil.make_archive("arc_lora_weights", 'zip', "./final_arc_adapter")
print("Archive created: arc_lora_weights.zip")

In [ ]:
# Inference script to generate predictions and clean GPU memory first
import json
import torch

print("Clearing GPU cache before inference...")
torch.cuda.empty_cache()

# Inline helper to avoid module import issues on Kaggle
def serialize_grid(grid):
    if not grid or not grid[0]:
        return ""
    return "\n".join(" ".join(str(cell) for cell in row) for row in grid)

test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"
try:
    with open(test_path, 'r') as f:
        test_tasks = json.load(f)
except FileNotFoundError:
    test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_training_challenges.json"
    with open(test_path, 'r') as f:
        test_tasks = json.load(f)

predictions = {}

# Convert to list first, THEN slice
task_items = list(test_tasks.items())[:5]

for task_id, task in task_items:
    prompt = "### Task Examples:\n"
    for idx, pair in enumerate(task["train"][:3]):
        prompt += f"Example {idx + 1}:\nInput:\n{serialize_grid(pair['input'])}\nOutput:\n{serialize_grid(pair['output'])}\n\n"

    test_input = task["test"][0]["input"]
    prompt += f"### Test Input:\n{serialize_grid(test_input)}\n\n### Predicted Output Grid:\n"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions[task_id] = decoded_output
    print(f"Finished predicting task: {task_id}")

# Save output to JSON
with open("predictions.json", "w") as f:
    json.dump(predictions, f, indent=4)

print("Inference complete")